# w1 · KKBOX churn · Ejercicios

12 ejercicios, de ⭐ a ⭐⭐⭐. Siete practican Polars y la construcción de features; cinco completan el split, la preparación y los modelos clásicos.

**Edita las funciones de [`src/estudio/ejercicios.py`](src/estudio/ejercicios.py)**. Los contratos de entrada/salida y las pistas están junto a cada función. El notebook importa, ejecuta y presenta resultados. Los `TODO` y los `assert` viven en `src`; la celda posterior a cada ejercicio llama a su verificación. Con la recarga automática puedes editar el `.py` y volver a ejecutar la celda sin reiniciar el kernel.

Empieza con `MODO = "mini"`: son datos **sintéticos** pequeños con el esquema KKBOX, duplicados, nulos y registros futuros. Para los datos originales usa `MODO = "real"`: se carga desde `workshops/data/`, igual que en w1, convirtiendo `.csv.7z` si hace falta. No hay que completar imports, carga ni plots. Usa las dependencias del proyecto; para esta práctica bastan numpy, pandas, polars, scikit-learn, xgboost, matplotlib e IPython/Jupyter.

Febrero es desarrollo (features hasta **2017-01-31**); marzo es test (hasta **2017-02-28**). El split interno de febrero separa usuarios. Un suscriptor puede aparecer de nuevo en marzo: el test mide otro mes para una población recurrente, como el proyecto original. Las features nunca consultan eventos posteriores a su cutoff y la preparación aprende solo con train. Los atributos de `members_v3` son una instantánea estática: sin historial no podemos reconstruir sus cambios pasados.

Esta práctica usa un subconjunto de features y presupuestos de entrenamiento pequeños. Optuna, refit y exportación siguen disponibles en los notebooks originales, que se conservan intactos. Los plots reutilizan sus funciones originales. No interpretes las métricas sintéticas como resultados reales de KKBOX.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# Funciona al abrir desde w1 o desde la raíz del repositorio.
W1_DIR = next(
    p for p in (Path.cwd(), Path.cwd() / "workshops" / "w1")
    if (p / "w1_data_preprocessing.ipynb").is_file()
)
sys.path.insert(0, str(W1_DIR.parent))
from w1.src.estudio import ejercicios as practica

from w1.src.estudio.contratos import CUTOFFS
from w1.src.estudio.soporte import (
    cargar_datos, crear_entradas, evaluar_test, graficar_resultados,
)
from w1.src.estudio import verificaciones as checks

MODO = "mini"
MAX_USUARIOS = 3000  # Por cohorte en modo real; None procesa todos.
CUTOFF = CUTOFFS["train_feb"]


In [ ]:
labels, members_raw, transactions_raw, logs_raw = cargar_datos(
    W1_DIR, modo=MODO, max_usuarios=MAX_USUARIOS
)


## 01 · ⭐ · Filtrar, seleccionar y deduplicar pagos

Conserva los pagos conocidos al cutoff **inclusive**. Selecciona `TXN_COLUMNS`, descarta copias exactas de negocio y retén la primera. `_ord` sirve para desempatar, no para identificar duplicados. Dos pagos del mismo día con distinto importe deben sobrevivir.

Función: `filtrar_transacciones` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué incluir `_ord` en la identidad impediría eliminar las copias exactas?

In [ ]:
transactions_cut = practica.filtrar_transacciones(transactions_raw, CUTOFF)
print(transactions_cut.shape)
transactions_cut.head(3)


In [ ]:
checks.verificar_01(practica.filtrar_transacciones)


## 02 · ⭐ · Crear columnas con expresiones Polars

Añade fechas reales de transacción y vencimiento, descuento y duración en días. Aprende a encadenar expresiones: una columna creada no se puede usar como si ya existiera dentro del mismo `with_columns`.

Función: `derivar_transacciones` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué restar dos enteros YYYYMMDD no mide días?

In [ ]:
transactions_derived = practica.derivar_transacciones(transactions_cut)
transactions_derived.head(3)


In [ ]:
checks.verificar_02(practica.derivar_transacciones)


## 03 · ⭐ · Limpiar nulos y contadores de escucha

Sustituye por cero los contadores negativos, nulos, NaN e infinitos. Conserva los valores válidos y todas las filas; no agregues todavía.

Función: `limpiar_logs` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Qué diferencia hay entre null y NaN en Polars?

In [ ]:
logs_clean = practica.limpiar_logs(logs_raw)
print(logs_clean.shape)
logs_clean.head(3)


In [ ]:
checks.verificar_03(practica.limpiar_logs)


## 04 · ⭐⭐ · Agregar transacciones por usuario

Obtén conteos, importes y recencias por usuario. El último pago se define por fecha y luego `_ord`; sus atributos deben pertenecer a la misma fila, incluso si llegan desordenadas.

Función: `agregar_transacciones` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué tomar el máximo independiente de cada columna no recupera el último pago?

In [ ]:
transaction_features = practica.agregar_transacciones(transactions_derived, CUTOFF)
print(transaction_features.shape)
transaction_features.head(3)


In [ ]:
checks.verificar_04(practica.agregar_transacciones)


## 05 · ⭐⭐ · Pasar de eventos a días y de días a usuarios

Usa los tres meses calendario que terminan en el cutoff. Primero combina duplicados por `(msno, date)` y después agrega por `msno`. La media por día activo debe dar el mismo peso a cada día, aunque tenga varios registros.

Función: `agregar_escucha` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Qué sesgo introduce promediar registros antes de consolidar cada día?

In [ ]:
daily, listening_features = practica.agregar_escucha(logs_clean, CUTOFF)
print("Días:", daily.shape, "| Usuarios:", listening_features.shape)
listening_features.head(3)


In [ ]:
checks.verificar_05(practica.agregar_escucha)


## 06 · ⭐⭐ · Limpiar miembros sin aprender estadísticas

Convierte edades y canales inválidos a nulos, conserva un indicador de edad ausente y deriva la antigüedad. Mantén las estadísticas pendientes hasta después del split. Una fecha de alta futura no produce una antigüedad negativa válida.

Función: `preparar_miembros` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué calcular la media de edad antes de dividir train/val sería una fuga?

In [ ]:
member_features = practica.preparar_miembros(members_raw, CUTOFF)
member_features.head(3)


In [ ]:
checks.verificar_06(practica.preparar_miembros)


## 07 · ⭐⭐ · Unir tablas y distinguir cero de desconocido

Parte de los usuarios etiquetados y conserva su orden con left joins. La ausencia de pagos implica conteo y suma cero, pero importe medio y fechas desconocidos. Añade indicadores de presencia y conserva nulos que el modelo debe imputar.

Función: `unir_features` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Qué usuarios se perderían con un inner join? ¿Qué confundirías al rellenar todos los nulos con cero?

In [ ]:
development = practica.unir_features(
    labels["train_feb"], member_features, transaction_features, listening_features
)
print("Desarrollo:", development.shape)
development.head(3)


In [ ]:
checks.verificar_07(practica.unir_features)


## 08 · ⭐⭐⭐ · Separar train/validación sin fuga de usuarios

Divide únicamente febrero 75/25, estratificando por churn y fijando la semilla. Cada usuario debe estar en una sola partición. Rechaza duplicados. Marzo se mantiene separado como test temporal.

Función: `dividir_desarrollo` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Qué garantiza la estratificación? ¿Qué cambiaría si hubiera varias filas por usuario en febrero?

In [ ]:
idx_train, idx_val = practica.dividir_desarrollo(development)
print("Train:", len(idx_train), "| Validación:", len(idx_val))


In [ ]:
checks.verificar_08(practica.dividir_desarrollo)


## 09 · ⭐⭐⭐ · Aprender imputación, escala y categorías solo en train

Construye un `ColumnTransformer`: mediana y escalado para numéricas; moda y one-hot para categorías. Aprende todo con train y transforma validación. Conserva columnas vacías y acepta categorías nuevas.

Función: `preparar_matrices` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Qué pasa si una categoría aparece por primera vez en validación?

In [ ]:
X_train, X_val, y_train, y_val = crear_entradas(development, idx_train, idx_val)
preprocessor, Z_train, Z_val = practica.preparar_matrices(X_train, X_val)
print("Matrices:", Z_train.shape, Z_val.shape)


In [ ]:
checks.verificar_09(practica.preparar_matrices)


## 10 · ⭐⭐⭐ · Entrenar la logística y un baseline de probabilidades

Ajusta un `DummyClassifier(strategy="prior")` y una regresión logística. El baseline debe predecir la prevalencia de **train**, aunque cambie la de validación. Devuelve probabilidades de churn, no etiquetas binarias.

Función: `entrenar_logistica_y_baseline` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué un accuracy alto puede ser engañoso cuando casi todos los usuarios no abandonan?

In [ ]:
models, proba_val = practica.entrenar_logistica_y_baseline(Z_train, y_train, Z_val)
print("Modelos ajustados:", list(models))


In [ ]:
checks.verificar_10(practica.entrenar_logistica_y_baseline)


## 11 · ⭐⭐⭐ · Entrenar Random Forest y XGBoost

Ajusta Random Forest y XGBoost con los parámetros indicados en `src`. Usa los mismos usuarios y matrices que la logística. Los hiperparámetros se fijan para concentrar la práctica en el entrenamiento y las probabilidades.

Función: `entrenar_arboles` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Por qué comparar modelos con diferentes splits puede llevarte a una conclusión falsa?

In [ ]:
tree_models, tree_proba = practica.entrenar_arboles(Z_train, y_train, Z_val)
models.update(tree_models)
proba_val.update(tree_proba)
print("Modelos ajustados:", list(models))


In [ ]:
checks.verificar_11(practica.entrenar_arboles)


## 12 · ⭐⭐⭐ · Evaluar desbalance y comparar contra el baseline

Calcula ROC-AUC y log loss usando probabilidades. Compara cada pérdida con la del Dummy y ordena la tabla. AUC mide ordenamiento; log loss también penaliza la confianza equivocada. Puedes aprobar aunque un modelo no gane al baseline.

Función: `evaluar_modelos` en [`ejercicios.py`](src/estudio/ejercicios.py).

Para razonar: ¿Puede mejorar AUC y empeorar log loss? Explica qué implicaría para las probabilidades.

In [ ]:
tabla_val = practica.evaluar_modelos(y_val, proba_val)
tabla_val.round(5)


In [ ]:
checks.verificar_12(practica.evaluar_modelos)


## Evaluación temporal y plots · código ya preparado

El ganador entre los tres modelos supervisados se fija por log loss en **validación**. La tabla también permite ver si el Dummy fue mejor. Después se transforman las features de marzo con la preparación de train y se evalúan los modelos ya ajustados. No se reajustan modelos, vocabularios ni umbrales con marzo. Los gráficos originales muestran ROC, PR y calibración; su generación no es un ejercicio.

Cambiar al modo real requiere volver a ejecutar desde la carga. Tras resolver todos los ejercicios, reinicia el kernel y ejecuta todo para comprobar que ninguna celda depende de un estado anterior.

In [ ]:
winner = tabla_val.loc[tabla_val["model"] != "baseline", "model"].iloc[0]
print("Ganador supervisado fijado en validación:", winner)
tabla_test, y_test, proba_test = evaluar_test(
    practica, preprocessor, models, winner,
    labels, members_raw, transactions_raw, logs_raw,
)
tabla_test.round(5)


In [ ]:
figures_dir = graficar_resultados(y_test, proba_test, winner, W1_DIR, MODO)
print("Figuras:", figures_dir)
